<a href="https://colab.research.google.com/github/pmatkowska94/MSC_Data_Science/blob/main/CHO_Augumentation%2BScaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
import joblib

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving test_df.csv to test_df.csv
Saving train_df.csv to train_df.csv
Saving val_df.csv to val_df.csv


In [ ]:
train_df = pd.read_csv('train_df.csv')
test_df = pd.read_csv('test_df.csv')
val_df = pd.read_csv('val_df.csv')

In [ ]:
train_df.head()

,p_id,r_id,clone_id,time,vcd,volume,p_v,vvm,tip_speed,reynolds,aspect_r,group_volume,seed_d,cell_type
0,1,1,1,-0.014656,0.68,10.0,100.0,0.07,1.03,44390.0,1.5,10.0,0.68,cho
1,1,1,1,0.996618,1.45,10.0,100.0,0.07,1.03,44390.0,1.5,10.0,0.68,cho
2,1,1,1,1.978579,3.75,10.0,100.0,0.07,1.03,44390.0,1.5,10.0,0.68,cho
3,1,1,1,2.989853,7.77,10.0,100.0,0.07,1.03,44390.0,1.5,10.0,0.68,cho
4,1,1,1,3.986471,12.52,10.0,100.0,0.07,1.03,44390.0,1.5,10.0,0.68,cho


In [ ]:
np.random.seed(42)

def augment_sequence (df_group, noise = 0.05):
  aug_df = df_group.copy()
  for col in ['p_v', 'vvm', 'tip_speed', 'reynolds']:
    aug_df[col] += aug_df[col] * np.random.uniform(-noise, noise)
  return aug_df

In [ ]:
train_df['group_id'] = train_df.groupby(['p_id', 'r_id', 'clone_id', 'group_volume']).ngroup()

train_aug = []

for group_id, df_group in train_df.groupby('group_id'):
    for i in range(2):
        augmented = augment_sequence(df_group)
        augmented['group_id'] = f'{group_id}_aug{i}'
        train_aug.append(augmented)
train_df['group_id'] = train_df['group_id'].astype(str)
train_aug = pd.concat([train_df] + train_aug, ignore_index=True)

train_aug.head()

,p_id,r_id,clone_id,time,vcd,volume,p_v,vvm,tip_speed,reynolds,aspect_r,group_volume,seed_d,cell_type,group_id
0,1,1,1,-0.014656,0.68,10.0,100.0,0.07,1.03,44390.0,1.5,10.0,0.68,cho,0
1,1,1,1,0.996618,1.45,10.0,100.0,0.07,1.03,44390.0,1.5,10.0,0.68,cho,0
2,1,1,1,1.978579,3.75,10.0,100.0,0.07,1.03,44390.0,1.5,10.0,0.68,cho,0
3,1,1,1,2.989853,7.77,10.0,100.0,0.07,1.03,44390.0,1.5,10.0,0.68,cho,0
4,1,1,1,3.986471,12.52,10.0,100.0,0.07,1.03,44390.0,1.5,10.0,0.68,cho,0


In [ ]:
train_aug['group_id'].nunique()

252

In [ ]:
train_df['group_id'].nunique()

84

In [ ]:
val_df['group_id'] = val_df.groupby(['p_id', 'r_id', 'clone_id', 'group_volume']).ngroup()
val_df['group_id'].nunique()

13

In [ ]:
test_df['group_id'] = test_df.groupby(['p_id', 'r_id', 'clone_id', 'group_volume']).ngroup()
test_df['group_id'].nunique()

17

In [ ]:
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
enc.fit(train_df[['clone_id']])

for df in [train_df, val_df, test_df]:
    df['clone_id'] = enc.transform(df[['clone_id']])
    df['clone_id'] = df['clone_id'].astype(int)

In [ ]:
def truncate_peak_plus_buffer(df, group_cols, buffer=2):
    d = df.sort_values(group_cols + ['time']).copy()
    out, removed = [], 0
    for _, g in d.groupby(group_cols, sort=False):
        peak_idx = g['vcd'].idxmax()
        peak_t = g.loc[peak_idx, 'time']
        kept = g[g['time'] <= peak_t + buffer]
        removed += len(g) - len(kept)
        out.append(kept)
    res = pd.concat(out, ignore_index=True)
    print(f"[truncate] before={len(d)} after={len(res)} removed={removed}")
    return res

group_cols = ['p_id','r_id','clone_id','group_volume', 'group_id']
val_df['group_id'] = 1
test_df['group_id'] = 1
train_aug = truncate_peak_plus_buffer(train_aug, group_cols, buffer=2)
val_df    = truncate_peak_plus_buffer(val_df,    group_cols, buffer=2)
test_df   = truncate_peak_plus_buffer(test_df,   group_cols, buffer=2)


[truncate] before=2430 after=2430 removed=0
[truncate] before=138 after=138 removed=0
[truncate] before=167 after=167 removed=0


In [ ]:
train_aug.to_csv('train_aug.csv', index=False)
files.download('train_aug.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving train_aug_ipsc.csv to train_aug_ipsc.csv


In [ ]:
train_aug_ipsc = pd.read_csv('train_aug_ipsc.csv')

In [ ]:
num_cols = ['time', 'vcd', 'volume', 'p_v', 'vvm', 'tip_speed', 'aspect_r', 'reynolds', 'seed_d']

scaling_df = pd.concat([train_aug[num_cols], train_aug_ipsc[num_cols]], ignore_index=True)

scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(scaling_df)

train_aug[num_cols] = scaler.transform(train_aug[num_cols])

joblib.dump(scaler, 'scaler_combined_cho_ipsc_main_study.pkl')

['scaler_combined_cho_ipsc_main_study.pkl']

In [ ]:
files.download('scaler_combined_cho_ipsc_main_study.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
category_df = pd.concat([train_aug[['cell_type']], train_aug_ipsc[['cell_type']]], ignore_index=True)

enc = OneHotEncoder(
    categories=[['CHO', 'iPSC']],
    drop=None,
    handle_unknown='ignore',
    sparse_output=False
)
enc.fit(category_df[['cell_type']])
Xohe_cho  = enc.transform(train_aug[['cell_type']])
Xohe_ipsc = enc.transform(train_aug_ipsc[['cell_type']])
ohe_cols = [f'cell_type_{c}' for c in enc.categories_[0]]
train_aug[ohe_cols]      = Xohe_cho
train_aug_ipsc[ohe_cols] = Xohe_ipsc
joblib.dump(enc, 'cell_type_enc.joblib')
files.download('cell_type_enc.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
test_df[num_cols] = scaler.transform(test_df[num_cols])
val_df[num_cols] = scaler.transform(val_df[num_cols])

In [ ]:
test_df[ohe_cols] = enc.transform(test_df[['cell_type']])
val_df[ohe_cols] = enc.transform(val_df[['cell_type']])

In [ ]:
val_df.head()

,p_id,r_id,clone_id,time,vcd,volume,p_v,vvm,tip_speed,reynolds,aspect_r,group_volume,seed_d,cell_type,group_id,cell_type_CHO,cell_type_iPSC
0,1,1,0,-0.992975,-0.982357,-0.000008,-0.853755,-0.997723,0.453876,-0.039005,-0.473684,1000.0,-0.706642,cho,1,0.0,0.0
1,1,1,0,-0.838678,-0.957514,-0.000008,-0.853755,-0.997723,0.453876,-0.039005,-0.473684,1000.0,-0.706642,cho,1,0.0,0.0
2,1,1,0,-0.688854,-0.879616,-0.000008,-0.853755,-0.997723,0.453876,-0.039005,-0.473684,1000.0,-0.706642,cho,1,0.0,0.0
3,1,1,0,-0.534558,-0.677922,-0.000008,-0.853755,-0.997723,0.453876,-0.039005,-0.473684,1000.0,-0.706642,cho,1,0.0,0.0
4,1,1,0,-0.382498,-0.421069,-0.000008,-0.853755,-0.997723,0.453876,-0.039005,-0.473684,1000.0,-0.706642,cho,1,0.0,0.0


In [ ]:
test_df.head()

,p_id,r_id,clone_id,time,vcd,volume,p_v,vvm,tip_speed,reynolds,aspect_r,group_volume,seed_d,cell_type,group_id,cell_type_CHO,cell_type_iPSC
0,3,1,2,-0.990739,-0.976883,-0.750013,-0.96695,-0.998546,-0.427752,-0.616971,-0.052632,250.0,-0.586716,cho,1,0.0,0.0
1,3,1,2,-0.838162,-0.961725,-0.750013,-0.96695,-0.998546,-0.427752,-0.616971,-0.052632,250.0,-0.586716,cho,1,0.0,0.0
2,3,1,2,-0.685586,-0.919196,-0.750013,-0.96695,-0.998546,-0.427752,-0.616971,-0.052632,250.0,-0.586716,cho,1,0.0,0.0
3,3,1,2,-0.533010,-0.864457,-0.750013,-0.96695,-0.998546,-0.427752,-0.616971,-0.052632,250.0,-0.586716,cho,1,0.0,0.0
4,3,1,2,-0.380434,-0.800454,-0.750013,-0.96695,-0.998546,-0.427752,-0.616971,-0.052632,250.0,-0.586716,cho,1,0.0,0.0


In [ ]:
train_aug.head()

,p_id,r_id,clone_id,time,vcd,volume,p_v,vvm,tip_speed,reynolds,aspect_r,group_volume,seed_d,cell_type,group_id,cell_type_CHO,cell_type_iPSC
0,1,1,1,-0.992975,-0.972672,-0.990015,-0.550395,-0.993216,-0.272741,-0.77876,-0.473684,10.0,-0.494465,cho,0,0.0,0.0
1,1,1,1,-0.838678,-0.940250,-0.990015,-0.550395,-0.993216,-0.272741,-0.77876,-0.473684,10.0,-0.494465,cho,0,0.0,0.0
2,1,1,1,-0.688854,-0.843403,-0.990015,-0.550395,-0.993216,-0.272741,-0.77876,-0.473684,10.0,-0.494465,cho,0,0.0,0.0
3,1,1,1,-0.534558,-0.674133,-0.990015,-0.550395,-0.993216,-0.272741,-0.77876,-0.473684,10.0,-0.494465,cho,0,0.0,0.0
4,1,1,1,-0.382498,-0.474124,-0.990015,-0.550395,-0.993216,-0.272741,-0.77876,-0.473684,10.0,-0.494465,cho,0,0.0,0.0


In [ ]:
train_aug.to_csv('train_aug.csv', index=False)
test_df.to_csv('test_df.csv', index=False)
val_df.to_csv('val_df.csv', index=False)

In [ ]:
files.download('train_aug.csv')
files.download('test_df.csv')
files.download('val_df.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>